In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sns
import warnings
import numpy as np
from datetime import date
from scipy import stats
import pandasql as ps
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('/Users/romandius/Python Works/Data Files/BA_ga4_events.csv') 

In [3]:
df.head()

,user_id,event_timestamp,event_name,product_id,product_category,session_id,device_type,purchase_value
0,U00031,2025-07-08T16:24:14.502920,purchase,P0058,Home,S35973,mobile,238.71
1,U01453,2025-05-22T16:31:14.905355,view_item,P0044,Books,S25975,mobile,NaN
2,U00124,2025-07-09T16:07:14.575552,view_item,P0080,Home,S36573,tablet,NaN
3,U00185,2025-07-07T16:21:14.617679,view_item,P0074,Electronics,S81191,tablet,NaN
4,U00024,2025-07-07T16:58:14.498248,view_item,P0053,Electronics,S41263,desktop,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50631 entries, 0 to 50630
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           50631 non-null  object 
 1   event_timestamp   50631 non-null  object 
 2   event_name        50631 non-null  object 
 3   product_id        50631 non-null  object 
 4   product_category  50631 non-null  object 
 5   session_id        50631 non-null  object 
 6   device_type       50631 non-null  object 
 7   purchase_value    1698 non-null   float64
dtypes: float64(1), object(7)
memory usage: 3.1+ MB


In [5]:
df.user_id.nunique()

2000

In [6]:
df.event_name.unique()

array(['purchase', 'view_item', 'add_to_cart', 'page_view'], dtype=object)

A lot of missed  data for column purchase_value

Customer Segmentation for Targeted Personalization
Data Analysis


Actionable Insights


Analyze and describe each segment:
- What are the defining characteristics of the different segments?
- Are segments clearly distinguishable?
- Propose website changes/implementations of new features tailored to each segment. Keep your suggestions brief.

Dashboard


Build a dashboard with a tool of your choice (e.g., Tableau, Power BI, Streamlit, Dash, Plotly) to:
- Visualize segment sizes
- Visualize feature distributions for each segment
- Show distribution of users across segments
- Top categories per segment


In [7]:
df = df.sort_values(by = ['session_id', 'user_id', 'event_timestamp'], ascending = True)

In [8]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'])

In [9]:
df.groupby('event_name').agg({'user_id': 'count'}).reset_index().sort_values(by='user_id', ascending = False)

,event_name,user_id
1,page_view,20224
3,view_item,18535
0,add_to_cart,10174
2,purchase,1698


So, we have 4 events in the table, funnel ideally should look like: 

**page_view** -> **view_item** -> **add_to_cart** -> **purchase**


What i see in the first rows, is that product id within the same session_id changes several times, thus i cannot be 100% sure whether funnel would be accurate. So, as i can build a proper funnel, i would calculate purchase CR based on proportion of purchases done out of all sessions  

In [10]:
df['event_order'] = df.groupby(['user_id', 'session_id', 'event_name']).cumcount()

pivot_df = df.pivot_table(
    index=['user_id', 'session_id', 'event_order', 'product_id', 'product_category', 'device_type'],
    columns='event_name',
    values='event_timestamp',
    aggfunc='first' 
).reset_index()

pivot_df = pivot_df.drop(columns='event_order')

pivot_df = pivot_df[['user_id', 'session_id', 'device_type', 'product_id', 'product_category',
                     'page_view', 'view_item', 'add_to_cart', 'purchase']]


In [11]:
filtered = pivot_df.loc[(pivot_df.page_view > pivot_df.add_to_cart) | 
    (pivot_df.page_view > pivot_df.purchase) |
    (pivot_df.view_item > pivot_df.add_to_cart) | 
    (pivot_df.view_item > pivot_df.purchase) |
    (pivot_df.add_to_cart > pivot_df.purchase) ]

i have 84 sessions with strange logic in them, so i want to eliminate them

In [12]:
filtered.session_id.values

array(['S69587', 'S34541', 'S89926', 'S57998', 'S94921', 'S63440',
       'S42314', 'S32222', 'S73486', 'S76885', 'S33278', 'S62605',
       'S31848', 'S37771', 'S30126', 'S98100', 'S75541', 'S99299',
       'S66294', 'S72673', 'S25960', 'S23523', 'S27569', 'S62682',
       'S66169', 'S39082', 'S44882', 'S44482', 'S12557', 'S88212',
       'S31905', 'S79954', 'S58112', 'S65008', 'S47395', 'S32369',
       'S35703', 'S67101', 'S28874', 'S27620', 'S41434', 'S53005',
       'S94330', 'S94646', 'S43899', 'S24370', 'S87241', 'S49451',
       'S83158', 'S77500', 'S25638', 'S31295', 'S83657', 'S96031',
       'S13586', 'S91507', 'S45626', 'S75822', 'S42015', 'S72261',
       'S84111', 'S61955', 'S79327', 'S96410', 'S66651', 'S94204',
       'S68210', 'S93984', 'S74857', 'S65398', 'S88920', 'S40521',
       'S44596', 'S79218', 'S81978', 'S37154', 'S44258', 'S53200',
       'S30931', 'S70802', 'S84315', 'S86178', 'S98763', 'S97868'],
      dtype=object)

In [13]:
df = df.loc[~df.session_id.isin(filtered.session_id)]

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50141 entries, 31839 to 12561
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           50141 non-null  object        
 1   event_timestamp   50141 non-null  datetime64[ns]
 2   event_name        50141 non-null  object        
 3   product_id        50141 non-null  object        
 4   product_category  50141 non-null  object        
 5   session_id        50141 non-null  object        
 6   device_type       50141 non-null  object        
 7   purchase_value    1666 non-null   float64       
 8   event_order       50141 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 3.8+ MB


I also want to check whether all events called purchase actually contain purchase value

In [15]:
df.loc[(df['event_name'] == 'purchase') & (df['event_name'].isna())]

,user_id,event_timestamp,event_name,product_id,product_category,session_id,device_type,purchase_value,event_order


Great, there are no such strange lines

In [16]:
df.groupby(['product_category', 'device_type']).agg(
    session_count=('session_id', 'count'),
    unique_sessions=('session_id', 'nunique'),
    unique_users=('user_id', 'nunique'),
    unique_products=('product_id', 'nunique'),
    product_counts=('product_id', 'count'),
    purchase_event_count=('purchase_value', 'count'),
    total_purchase_value=('purchase_value', 'sum')
).reset_index()

,product_category,device_type,session_count,unique_sessions,unique_users,unique_products,product_counts,purchase_event_count,total_purchase_value
0,Books,desktop,372,88,88,98,372,0,0.00
1,Books,mobile,313,72,72,97,313,0,0.00
2,Books,tablet,335,77,77,97,335,0,0.00
3,Electronics,desktop,8277,1811,733,100,8277,364,48998.86
4,Electronics,mobile,8890,1938,754,100,8890,335,48312.88
5,Electronics,tablet,8409,1853,736,100,8409,385,54472.35
6,Fashion,desktop,1621,359,295,100,1621,0,0.00
7,Fashion,mobile,1434,332,281,100,1434,0,0.00
8,Fashion,tablet,1651,366,293,100,1651,0,0.00
9,Garden,desktop,430,94,94,100,430,0,0.00


In [17]:
df.groupby(df['event_timestamp'].dt.strftime('%Y-%m-%u')).agg({'user_id': 'count'})

,user_id
event_timestamp,
2025-05-1,45
2025-05-2,40
2025-05-3,47
2025-05-4,42
2025-05-5,55
2025-05-6,72
2025-05-7,22
2025-06-1,1112
2025-06-2,480


we have timestamps for approximately 3 months so better would be split everything into weekly periods, if we want to trim dates

In [18]:
df['event_timestamp'] = pd.to_datetime(df['event_timestamp'], format='%Y-%m-%d %H:%M:%S.%f')
df = df.sort_values(by=['user_id', 'event_timestamp'])
session_start = df.groupby(['user_id', 'session_id'])['event_timestamp'].min().reset_index()
session_start = session_start.sort_values(by=['user_id', 'event_timestamp'])

session_start['session_rank'] = session_start.groupby('user_id').cumcount() + 1


session_purchases = df.groupby(['user_id', 'session_id'])['purchase_value'].apply(lambda x: x.notna().any()).reset_index(name='purchase_made')

session_meta = session_start.merge(session_purchases, on=['user_id', 'session_id'])

session_meta['past_purchases'] = (
    session_meta.groupby('user_id')['purchase_made']
    .cumsum()
    .shift(fill_value=0)
)

def classify_session(row):
    if row['session_rank'] == 1:
        return 'new_user'
    elif row['past_purchases'] == 0:
        return 'returning'
    else:
        return 'made_purchase_already'

session_meta['user_classification'] = session_meta.apply(classify_session, axis=1)

metrics = df.groupby(['user_id', 'session_id', 'device_type']).agg(
    first_session_event_timestamp=('event_timestamp', 'min'),
    last_session_event_timestamp=('event_timestamp', 'max'),
    purchase_total=('purchase_value', 'sum'),
    events_detected=('event_name', 'count'),
    unique_products_viewed=('product_id', pd.Series.nunique),
    unique_products_categories_viewed=('product_category', pd.Series.nunique)
).reset_index()

session_summary = metrics.merge(
    session_meta[['user_id', 'session_id', 'user_classification']],
    on=['user_id', 'session_id'],
    how='left'
)



In [19]:
session_summary.loc[session_summary.user_id == 'U00001'].sort_values(by= 'first_session_event_timestamp')

,user_id,session_id,device_type,first_session_event_timestamp,last_session_event_timestamp,purchase_total,events_detected,unique_products_viewed,unique_products_categories_viewed,user_classification
8,U00001,S42246,mobile,2025-07-06 16:09:14.482380,2025-07-06 16:19:14.482380,0.00,3,3,1,new_user
9,U00001,S63904,mobile,2025-07-06 16:17:14.482373,2025-07-06 17:06:14.482373,203.92,5,5,1,returning
5,U00001,S27393,mobile,2025-07-06 16:20:14.482365,2025-07-06 16:59:14.482365,0.00,4,4,1,made_purchase_already
6,U00001,S41892,mobile,2025-07-06 16:20:14.482368,2025-07-06 16:59:14.482368,0.00,3,3,1,made_purchase_already
1,U00001,S11641,mobile,2025-07-06 16:23:14.482377,2025-07-06 16:46:14.482377,0.00,5,5,1,made_purchase_already
12,U00001,S78203,tablet,2025-07-07 16:15:14.482372,2025-07-07 17:02:14.482372,0.00,7,7,1,made_purchase_already
7,U00001,S42150,tablet,2025-07-07 16:21:14.482371,2025-07-07 17:04:14.482371,131.21,4,4,1,made_purchase_already
3,U00001,S21848,desktop,2025-07-07 16:22:14.482381,2025-07-07 17:01:14.482381,0.00,5,5,1,made_purchase_already
0,U00001,S10971,tablet,2025-07-08 16:07:14.482378,2025-07-08 16:51:14.482378,0.00,2,2,1,made_purchase_already
15,U00001,S88314,mobile,2025-07-08 16:13:14.482377,2025-07-08 17:03:14.482377,0.00,7,7,1,made_purchase_already


aggregation looks correct, now i want to check this ambuguiness with unique_products_categories_viewed

In [20]:
session_summary.groupby('session_id').agg({'unique_products_categories_viewed': 'max'}).reset_index().sort_values(by='unique_products_categories_viewed', ascending = False )

,session_id,unique_products_categories_viewed
0,S10008,1
6989,S69953,1
6971,S69799,1
6972,S69810,1
6973,S69815,1
...,...,...
3489,S40125,1
3490,S40130,1
3491,S40144,1
3492,S40151,1


so, we have no sessions with multiple product categories viewed and so we can use as one of the dimensions 

In [21]:
metrics = df.groupby(['user_id', 'session_id', 'device_type', 'product_category']).agg(
    first_session_event_timestamp=('event_timestamp', 'min'),
    last_session_event_timestamp=('event_timestamp', 'max'),
    purchase_total=('purchase_value', 'sum'),
    events_detected=('event_name', 'count'),
    unique_products_viewed=('product_id', pd.Series.nunique),
).reset_index()

session_summary = metrics.merge(
    session_meta[['user_id', 'session_id', 'user_classification']],
    on=['user_id', 'session_id'],
    how='left'
)

In [22]:
session_summary['session_duration'] = (session_summary['last_session_event_timestamp'] - session_summary['first_session_event_timestamp']) / pd.Timedelta(minutes = 1)

In [23]:
session_summary['session_date'] = session_summary['first_session_event_timestamp'].dt.strftime('%Y-%m-%d')

In [24]:
session_summary['session_week'] = session_summary['first_session_event_timestamp'].dt.strftime('%Y-%m-%u')

In [25]:
session_summary.head().sort_values(by= ['first_session_event_timestamp', 'user_id'], ascending = True)

,user_id,session_id,device_type,product_category,first_session_event_timestamp,last_session_event_timestamp,purchase_total,events_detected,unique_products_viewed,user_classification,session_duration,session_date,session_week
1,U00001,S11641,mobile,Electronics,2025-07-06 16:23:14.482377,2025-07-06 16:46:14.482377,0.0,5,5,made_purchase_already,23.0,2025-07-06,2025-07-7
3,U00001,S21848,desktop,Home,2025-07-07 16:22:14.482381,2025-07-07 17:01:14.482381,0.0,5,5,made_purchase_already,39.0,2025-07-07,2025-07-1
0,U00001,S10971,tablet,Home,2025-07-08 16:07:14.482378,2025-07-08 16:51:14.482378,0.0,2,2,made_purchase_already,44.0,2025-07-08,2025-07-2
2,U00001,S13790,tablet,Electronics,2025-07-09 16:06:14.482376,2025-07-09 16:47:14.482376,240.0,3,3,made_purchase_already,41.0,2025-07-09,2025-07-3
4,U00001,S26037,desktop,Electronics,2025-07-10 16:22:14.482370,2025-07-10 16:58:14.482370,0.0,5,5,made_purchase_already,36.0,2025-07-10,2025-07-4


In [26]:
first_grouped = session_summary.groupby('user_id').agg(first_session_week=('session_week', 'min')).reset_index()

In [27]:
first_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_id             1995 non-null   object
 1   first_session_week  1995 non-null   object
dtypes: object(2)
memory usage: 31.3+ KB


In [28]:
session_summary['user_id'] = session_summary['user_id'].str.strip()
first_grouped['user_id'] = first_grouped['user_id'].str.strip()

session_summary = session_summary.merge(
    first_grouped[['user_id', 'first_session_week']],
    on='user_id',
    how='left'
)

In [29]:
session_summary.head()

,user_id,session_id,device_type,product_category,first_session_event_timestamp,last_session_event_timestamp,purchase_total,events_detected,unique_products_viewed,user_classification,session_duration,session_date,session_week,first_session_week
0,U00001,S10971,tablet,Home,2025-07-08 16:07:14.482378,2025-07-08 16:51:14.482378,0.0,2,2,made_purchase_already,44.0,2025-07-08,2025-07-2,2025-07-1
1,U00001,S11641,mobile,Electronics,2025-07-06 16:23:14.482377,2025-07-06 16:46:14.482377,0.0,5,5,made_purchase_already,23.0,2025-07-06,2025-07-7,2025-07-1
2,U00001,S13790,tablet,Electronics,2025-07-09 16:06:14.482376,2025-07-09 16:47:14.482376,240.0,3,3,made_purchase_already,41.0,2025-07-09,2025-07-3,2025-07-1
3,U00001,S21848,desktop,Home,2025-07-07 16:22:14.482381,2025-07-07 17:01:14.482381,0.0,5,5,made_purchase_already,39.0,2025-07-07,2025-07-1,2025-07-1
4,U00001,S26037,desktop,Electronics,2025-07-10 16:22:14.482370,2025-07-10 16:58:14.482370,0.0,5,5,made_purchase_already,36.0,2025-07-10,2025-07-4,2025-07-1


In [30]:
pivot_df = pivot_df.loc[~pivot_df.session_id.isin(filtered.session_id)]

In [31]:
pivot_df.head()

event_name,user_id,session_id,device_type,product_id,product_category,page_view,view_item,add_to_cart,purchase
0,U00001,S10971,tablet,P0031,Home,2025-07-08 16:51:14.482378,NaT,NaT,NaT
1,U00001,S10971,tablet,P0098,Home,NaT,NaT,2025-07-08 16:07:14.482378,NaT
2,U00001,S11641,mobile,P0026,Electronics,NaT,2025-07-06 16:37:14.482377,NaT,NaT
3,U00001,S11641,mobile,P0078,Electronics,2025-07-06 16:23:14.482377,NaT,NaT,NaT
4,U00001,S11641,mobile,P0056,Electronics,2025-07-06 16:26:14.482377,NaT,NaT,NaT


In [32]:
agg_df = pivot_df.groupby(['user_id', 'session_id', 'device_type']).agg(
    page_view_first=('page_view', 'min'),
    page_view_count=('page_view', lambda x: x.notna().sum()),

    view_item_first=('view_item', 'min'),
    view_item_count=('view_item', lambda x: x.notna().sum()),

    add_to_cart_first=('add_to_cart', 'min'),
    add_to_cart_count=('add_to_cart', lambda x: x.notna().sum()),

    purchase_first=('purchase', 'min'),
    purchase_count=('purchase', lambda x: x.notna().sum())
).reset_index()

In [33]:
agg_df.head()

,user_id,session_id,device_type,page_view_first,page_view_count,view_item_first,view_item_count,add_to_cart_first,add_to_cart_count,purchase_first,purchase_count
0,U00001,S10971,tablet,2025-07-08 16:51:14.482378,1,NaT,0,2025-07-08 16:07:14.482378,1,NaT,0
1,U00001,S11641,mobile,2025-07-06 16:23:14.482377,4,2025-07-06 16:37:14.482377,1,NaT,0,NaT,0
2,U00001,S13790,tablet,2025-07-09 16:06:14.482376,2,NaT,0,NaT,0,2025-07-09 16:47:14.482376,1
3,U00001,S21848,desktop,2025-07-07 16:22:14.482381,4,NaT,0,2025-07-07 17:01:14.482381,1,NaT,0
4,U00001,S26037,desktop,2025-07-10 16:22:14.482370,3,2025-07-10 16:23:14.482370,2,NaT,0,NaT,0


In [34]:
agg_df['user_id'] = agg_df['user_id'].str.strip()
session_summary = session_summary.merge(
    agg_df[['user_id', 'session_id', 'device_type',
            'page_view_first', 'page_view_count',
            'view_item_first', 'view_item_count',
            'add_to_cart_first', 'add_to_cart_count',
            'purchase_first', 'purchase_count']],
    on=['user_id', 'session_id', 'device_type'],
    how='left'
)

in half od cases we have strange rates

In [35]:
pivot_df.head()

event_name,user_id,session_id,device_type,product_id,product_category,page_view,view_item,add_to_cart,purchase
0,U00001,S10971,tablet,P0031,Home,2025-07-08 16:51:14.482378,NaT,NaT,NaT
1,U00001,S10971,tablet,P0098,Home,NaT,NaT,2025-07-08 16:07:14.482378,NaT
2,U00001,S11641,mobile,P0026,Electronics,NaT,2025-07-06 16:37:14.482377,NaT,NaT
3,U00001,S11641,mobile,P0078,Electronics,2025-07-06 16:23:14.482377,NaT,NaT,NaT
4,U00001,S11641,mobile,P0056,Electronics,2025-07-06 16:26:14.482377,NaT,NaT,NaT


In [36]:
new_pivot = pivot_df.groupby('user_id').agg(
    devices_used = ('device_type', 'nunique'), 
    product_categories_interested = ('product_category', 'nunique'), 
    product_interested = ('product_id', 'nunique'), 
    purchases_completed = ('purchase', 'count'), 
    sessions_created = ('session_id', 'nunique'),
).reset_index()

In [37]:
new_pivot['purchases_per_sessions'] = new_pivot['purchases_completed'] / new_pivot['sessions_created']
new_pivot['purchases_per_products_viewed'] = new_pivot['purchases_completed'] / new_pivot['product_interested']
new_pivot = new_pivot.round(2)

In [38]:
new_pivot.head()

,user_id,devices_used,product_categories_interested,product_interested,purchases_completed,sessions_created,purchases_per_sessions,purchases_per_products_viewed
0,U00001,3,2,52,4,16,0.25,0.08
1,U00002,3,2,55,3,17,0.18,0.05
2,U00003,3,2,56,3,16,0.19,0.05
3,U00004,3,2,53,2,16,0.12,0.04
4,U00005,3,2,50,1,17,0.06,0.02


In [39]:
new_pivot.describe()

,devices_used,product_categories_interested,product_interested,purchases_completed,sessions_created,purchases_per_sessions,purchases_per_products_viewed
count,1995.000000,1995.000000,1995.000000,1995.000000,1995.000000,1995.000000,1995.000000
mean,2.131328,1.445614,20.353885,0.835088,5.585464,0.078657,0.022301
std,0.847319,0.497158,16.254683,1.470326,5.078628,0.136906,0.038348
min,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000
25%,1.000000,1.000000,7.000000,0.000000,2.000000,0.000000,0.000000
50%,2.000000,1.000000,15.000000,0.000000,4.000000,0.000000,0.000000
75%,3.000000,2.000000,31.000000,1.000000,8.000000,0.130000,0.040000
max,3.000000,2.000000,68.000000,9.000000,19.000000,0.800000,0.210000


Most users used 1–2 devices, viewed 15–20 products, and browsed 1 category.

Median users did not complete a purchase, but ~42% did at least one (75th percentile = 1).

Only a small portion of views turned into purchases: ~2% conversion on average.

segmentation logic: 

| Segment Name                  | Description                                                                  | Defining Characteristics                                                                                          |
| ----------------------------- | ---------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------- |
| **1. Explorers**              | Curious users who deeply browse but never convert                            | - **0 purchases** <br> - High product views (`>25`)  <br> - High sessions (`>6`)                                  |
| **2. Engaged Non-Converters** | Users who show moderate to high engagement but don’t buy                     | - **0 purchases** <br> - Product views (`>15`) <br> - Sessions (`4–6`)                                            |
| **3. Casual Browsers**        | Users with light engagement and no purchases                                 | - **0 purchases** <br> - Product views (`<15`) <br> - Sessions (`≤3`)                                             |
| **4. Intentional Buyers**           | Efficient and intentional buyers                                             | - **≥1 purchase** <br> - Product views (`10–30`) <br> - High purchase/session ratio (`>0.13`)                        |
| **5. Casual Buyers**           | Buyers who convert but with low frequency or unclear intent                  | - **≥1 purchase** <br> - Not meeting Intentional Buyer criteria                                                         |
                  


In [40]:
def classify_user(row):
    if row['purchases_completed'] == 0:
        if row['product_interested'] > 25 and row['sessions_created'] > 6:
            return 'Explorers'
        elif row['product_interested'] > 15 and 4 <= row['sessions_created'] <= 6:
            return 'Engaged Non-Converters'
        else:
            return 'Casual Browsers'
    else:
        if 10 <= row['product_interested'] <= 30 and row['purchases_per_sessions'] > 0.13:
            return 'Intentional Buyers'
        else:
            return 'Casual Buyers'

new_pivot['user_segment'] = new_pivot.apply(classify_user, axis=1)


## 🔍 Segment Analysis & Actionable Insights

### **1. Explorers**
- **Defining Characteristics**:  
  - **0 purchases**  
  - High product views (`>25`)  
  - High sessions (`>6`)
- **Are They Distinct?**  
  ✅ Very distinct — heavy engagement, no conversion
- **Website Suggestions**:
  - Add “Save for Later” or Wishlist functionality  
  - Trigger discounts or limited-time offer popups  
  - Offer product comparison or filtering tools

---

### **2. Engaged Non-Converters**
- **Defining Characteristics**:  
  - **0 purchases**  
  - Product views (`>15`)  
  - Sessions (`4–6`)
- **Are They Distinct?**  
  ✅ Distinct — clearly engaged but not converting
- **Website Suggestions**:
  - Use exit-intent popups with discount codes  
  - Send browse-abandonment emails  
  - Add social proof (e.g., reviews, popularity badges)

---

### **3. Casual Browsers**
- **Defining Characteristics**:  
  - **0 purchases**  
  - Product views (`<15`)  
  - Sessions (`≤3`)
- **Are They Distinct?**  
  ✅ Clear — low activity and disengagement
- **Website Suggestions**:
  - Simplify homepage and product discovery  
  - Retarget with onboarding emails or ads  
  - Highlight best-sellers and quick-buy options

---

### **4. Intentional Buyers**
- **Defining Characteristics**:  
  - **≥1 purchase**  
  - Product views (`10–30`)  
  - High purchase-to-session ratio (`>0.13`)
- **Are They Distinct?**  
  ✅ Yes — show focused and efficient purchase behavior
- **Website Suggestions**:
  - Offer loyalty points or personalized deals  
  - Recommend products based on past behavior  
  - Provide early access to new or exclusive items

---

### **5. Casual Buyers**
- **Defining Characteristics**:  
  - **≥1 purchase**  
  - Do not meet Intentional Buyer criteria
- **Are They Distinct?**  
  ⚠️ Somewhat broad — catch-all for less consistent buyers
- **Website Suggestions**:
  - Use post-purchase emails to drive repeat buying  
  - Recommend bundles or limited-time upsells  
  - Highlight benefits of creating an account or subscribing


In [41]:
new_pivot.head()

,user_id,devices_used,product_categories_interested,product_interested,purchases_completed,sessions_created,purchases_per_sessions,purchases_per_products_viewed,user_segment
0,U00001,3,2,52,4,16,0.25,0.08,Casual Buyers
1,U00002,3,2,55,3,17,0.18,0.05,Casual Buyers
2,U00003,3,2,56,3,16,0.19,0.05,Casual Buyers
3,U00004,3,2,53,2,16,0.12,0.04,Casual Buyers
4,U00005,3,2,50,1,17,0.06,0.02,Casual Buyers


In [42]:
new_pivot['user_id'] = new_pivot['user_id'].str.strip()
session_summary = session_summary.merge(
    new_pivot[['user_id', 'user_segment']],
    on='user_id',
    how='left'
)

In [43]:
summary = session_summary.groupby(['session_date', 'device_type', 'product_category', 'user_segment']).agg(
    unique_users=('user_id', 'nunique'),
    unique_sessions=('session_id', 'nunique'),
    homepage_views=('page_view_count', 'sum'),
    view_item_views=('view_item_count', 'sum'),
    add_to_cart_views=('add_to_cart_count', 'sum'),
    purchase_views=('purchase_count', 'sum'),
    avg_session_duration_minutes=('session_duration', 'mean'),
    total_purchase_value=('purchase_total', 'sum'),
).reset_index()

summary['click_through_rate'] = summary['view_item_views'] / summary['homepage_views']
summary['add_to_cart_rate'] = summary['add_to_cart_views'] / summary['view_item_views']
summary['order_conversion_rate'] = summary['purchase_views'] / summary['add_to_cart_views']
summary['overall_conversion_rate'] = summary['purchase_views'] / summary['homepage_views']

summary = summary.round(2)

In [44]:
summary.loc[(summary['click_through_rate'] > 1) | 
    (summary['add_to_cart_rate'] > 1) | 
    (summary['order_conversion_rate'] > 1) | 
    (summary['overall_conversion_rate'] > 1)].shape

(384, 16)

In [45]:
summary.head()

,session_date,device_type,product_category,user_segment,unique_users,unique_sessions,homepage_views,view_item_views,add_to_cart_views,purchase_views,avg_session_duration_minutes,total_purchase_value,click_through_rate,add_to_cart_rate,order_conversion_rate,overall_conversion_rate
0,2025-05-12,tablet,Books,Casual Browsers,1,1,1,3,2,0,46.0,0.0,3.00,0.67,0.0,0.0
1,2025-05-13,mobile,Books,Casual Browsers,1,1,1,0,1,0,3.0,0.0,0.00,inf,0.0,0.0
2,2025-05-13,tablet,Books,Casual Browsers,1,1,3,0,0,0,51.0,0.0,0.00,NaN,NaN,0.0
3,2025-05-15,desktop,Books,Casual Browsers,1,1,1,2,0,0,21.0,0.0,2.00,0.00,NaN,0.0
4,2025-05-19,tablet,Garden,Casual Browsers,1,1,3,2,0,0,39.0,0.0,0.67,0.00,NaN,0.0


In [46]:
summary.to_csv('/Users/romandius/Python Works/Data Files/summary.csv', index=False) 

In [47]:

df = df.merge(
    new_pivot[['user_id', 'user_segment']],
    on='user_id',
    how='left'
)

In [48]:
df.to_csv('/Users/romandius/Python Works/Data Files/og_table.csv', index=False) 

In [49]:
df.product_category.unique()

array(['Electronics', 'Home', 'Fashion', 'Toys', 'Books', 'Garden'],
      dtype=object)

In [54]:
summary.groupby('user_segment').agg(
    purchase_count = ('purchase_views', 'sum'),
    session = ('unique_sessions', 'sum'),
)

,purchase_count,session
user_segment,,
Casual Browsers,0,2081
Casual Buyers,1357,6336
Engaged Non-Converters,0,859
Explorers,0,813
Intentional Buyers,309,1049
